In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

In [2]:
a2 = np.array([[2.3, 4.3]])

w3 = np.array([[0.45, 0.533], [0.23, 0.78]])

y = np.array([[0, 1]])

z3 = a2@w3

a3 = np.exp(z3)/ np.sum(np.exp(z3))

dz3 = a3 - y

dw3 = a2.T@dz3


In [3]:
class ClassificationNeuralNetwork():
    
    def __init__(self, input):
        
        # layer 1 weights & bias
        self.W1 = np.random.randn(input, 3)
        self.b1 = np.zeros((1, 3))
        
        # layer 2 weights & bias
        self.W2 = np.random.randn(3, 2)
        self.b2 = np.zeros((1, 2))
        
        # output layer weights & bias
        self.W3 = np.random.randn(2, 2)
        self.b3 = np.zeros((1, 2))
        
    def forward(self, X):
        
        # layer 1 
        self.z1 = X@self.W1 + self.b1
        self.a1 = np.tanh(self.z1) # tanh as activation
        
        # layer 2
        self.z2 = self.a1@self.W2 + self.b2
        self.a2 = np.tanh(self.z2) # tanh as activation
        
        # layer 3 
        self.z3 = self.a2@self.W3 + self.b3
        self.a3 = np.exp(self.z3)/ np.sum(np.exp(self.z3), axis=1, keepdims=True) #axis=1, keepdims=True
        
    def backward(self, X, y):
        
        # layer 3
        self.dz3 = np.asarray(self.a3 - y)
        self.dw3 = self.a2.T@self.dz3
        #print(type(self.dz3), type(self.W3), type(self.a2))
        
        # layer 2
        self.dz2 = (self.dz3@self.W3.T) * (1 - self.a2**2)
        self.dw2 = self.a1.T@self.dz2
        
        # layer 1
        self.dz1 = (self.dz2@self.W2.T) * (1 - (self.a1)**2)
        self.dw1 = X.T@self.dz1
        #print(self.dw1.shape)
        
        
    def update(self, lr, batch_size):
        
        # update weights
        self.W1 -= (self.dw1 * lr)/ batch_size
        self.W2 -= (self.dw2 * lr)/ batch_size
        self.W3 -= (self.dw3 * lr)/batch_size
        
        # update biases
        self.b1 -= np.sum(self.dz1, axis=0).reshape(1, -1) * lr
        self.b2 -= np.sum(self.dz2, axis=0).reshape(1, -1) * lr
        self.b3 -= np.sum(self.dz3, axis=0).reshape(1, -1) * lr

    def fit(self, X, y, n_epochs, lr):
        
        for epoch in range(n_epochs):
            self.forward(X)
            self.backward(X, y)
            self.update(lr, X.shape[0])
            
        
    def predict(self, X):
        self.forward(X)#return (self.a3 >= threshold).astype(int)
        return self.a3

Testing on real data

In [4]:
from ucimlrepo import fetch_ucirepo 
  
# fetching classification dataset 
toxicity = fetch_ucirepo(id=728) 
  
# data (as pandas dataframes) 
X_toxic = toxicity.data.features 
y_toxic = toxicity.data.targets 

# for simplicity sake we are skipping all the categorical features and considering only around 10 random features

toxicity_features = ['MATS3v', 'MATS3s', 'MATS3p', 'nHBDon_Lipinski', 'minHBint8', 'MATS3e', 'MATS3c', 'MATS3m'] 
X_toxic = X_toxic[toxicity_features]

X_toxic = np.array(X_toxic)
y_toxic = (np.array(y_toxic) != "NonToxic").astype(int).reshape(-1, 1)

In [5]:
# one hot encoding of labels
from sklearn.preprocessing import OneHotEncoder as ohe

ohe = ohe()
y_toxic_encoded = ohe.fit_transform(y_toxic)
y_toxic_encoded.shape

(171, 2)

In [12]:
cls_model_real = ClassificationNeuralNetwork(X_toxic.shape[-1])
cls_model_real.fit(X_toxic, y_toxic_encoded, 150, 0.001)
y_cls_model_real = np.argmax(cls_model_real.predict(X_toxic), axis=1).reshape(-1, 1)

In [13]:
print(classification_report(y_toxic, y_cls_model_real))

              precision    recall  f1-score   support

           0       0.67      0.94      0.78       115
           1       0.22      0.04      0.06        56

    accuracy                           0.64       171
   macro avg       0.44      0.49      0.42       171
weighted avg       0.52      0.64      0.54       171

